# TODO:
1. Save processed sdata file to original zarr file

# Packages

In [1]:
# import cellcharter

import cell2location

import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

/Users/janzules/miniforge3/envs/spatial_cpu_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import spatialdata as spd
import anndata
import scanpy as sc

/Users/janzules/miniforge3/envs/spatial_cpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/janzules/miniforge3/envs/spatial_cpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
import pandas as pd
import os
import mygene # Grabbing ENSEMBLE ID

In [4]:
# Run only on the HPC CUDA node
# import torch
# print(f"Cuda available? {torch.cuda.is_available()}")
# print(f"Cuda device: {torch.cuda.get_device_name(0)}")

# Functions

In [4]:
def del_DS_Store(root):
    """
    Recursively delete all .DS_Store files under 'root'.
    and read zarr.
    """
    root = Path(root)

    for p in root.rglob(".DS_Store"):
        p.unlink()


    

# Data

## Laptop

In [5]:
# Locations
proj_folder  = Path("/Users/janzules/Roselab/Spatial/CAR_T/")
data_folder  = proj_folder / "data"
zarr_loc     = data_folder / "zarrFiles/CART_centroid"
sc_ref_loc   = data_folder / "sc-reference/sc_reference_cell2location.h5ad"

# Output location
results_folder = proj_folder / "Results/cell2location"
ref_run_name   = results_folder / "reference_signatures"
run_name       = results_folder / "cell2location_map"

In [7]:
# for c in zarr_loc.rglob(".DS_Store"):
#     print(c)
#     print(f"Is this a file? {c.is_file()}")

In [8]:
# os.mkdir(ref_run_name)
# os.mkdir(run_name)

In [6]:
# Loading
# del_DS_Store(zarr_loc) # Removing random .DS_Store added to file tree because Macbook

zarr_spatial = spd.read_zarr(zarr_loc)
sdata        = zarr_spatial.tables['segmentation_counts']
del zarr_spatial # Don't need for now

adata_ref    = sc.read_h5ad(sc_ref_loc)

/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_1987/3877133078.py:4: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  zarr_spatial = spd.read_zarr(zarr_loc)


# Pre processing

In [7]:
sdata.obs.head()

,sample,cell_id,region,TMA,mouse,tissue,condition,tumor_loc,replicate_num
F07839_cellid_000000001-1,F07839,F07839_cellid_000000001-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1
F07839_cellid_000000002-1,F07839,F07839_cellid_000000002-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1
F07839_cellid_000000004-1,F07839,F07839_cellid_000000004-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1
F07839_cellid_000000005-1,F07839,F07839_cellid_000000005-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1
F07839_cellid_000000006-1,F07839,F07839_cellid_000000006-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1


## Compatible Naming Schemes

In [8]:
# Saving as integer
sdata.obs['tumor_loc'] = sdata.obs['tumor_loc'].astype(int)
sdata.obs['replicate_num'] = sdata.obs['replicate_num'].astype(int)

In [9]:
# Creating the column that will match the sc reference
sdata.obs['treatment'] = sdata.obs['condition']

tumor_locations = [1, 2]

for tumor in tumor_locations:

    if tumor == 1:
        tumor_num = 1
        suffix = "_Tu1"
    elif tumor == 2:
        tumor_num = 2
        suffix = "_Tu2"
    else:
        ValueError("Something went wrong")
        
    tum_loc_mask = sdata.obs["tumor_loc"] == tumor_num
    
    sdata.obs.loc[tum_loc_mask, 'treatment'] = (
        sdata.obs.loc[tum_loc_mask, 'treatment']
        .str.replace(r"T72", "TAG72")
        + suffix
    )
# sdata.obs['treatment']


In [10]:
sdata.obs['treatment'].unique()

array(['CyPSCA_Tu1', 'CyPSCA_Tu2', 'CyTAG72_Tu1', 'CyTAG72_Tu2',
       'NoTx_Tu1', 'NoTx_Tu2', 'RTCyPSCA_Tu1', 'RTCyPSCA_Tu2',
       'RTCyTAG72_Tu2', 'RTCyTAG72_Tu1'], dtype=object)

### SC - Names 

#### Checking for dissimilarity

In [16]:
sdata_trmts = set(sdata.obs['treatment'].astype(str))
adata_trmts = set(adata_ref.obs['treatment'].astype(str))

In [17]:
for name in sorted(sdata_trmts):
    print(name)

print(f"Length of names: {len(sdata_trmts)}")

CyPSCA_Tu1
CyPSCA_Tu2
CyTAG72_Tu1
CyTAG72_Tu2
NoTx_Tu1
NoTx_Tu2
RTCyPSCA_Tu1
RTCyPSCA_Tu2
RTCyTAG72_Tu1
RTCyTAG72_Tu2
Length of names: 10


In [18]:
for name in sorted(adata_trmts):
    print(name)

print(f"Length of names: {len(adata_trmts)}")

CyPSCA_Tu1
CyPSCA_Tu2
CyTAG72_Tu1
CyTAG72_Tu2
NoTx_Tu1
NoTx_Tu2
RTCyPSCA_Tu1
RTCyPSCA_Tu2
RTCyTAG72_Tu1
RTCyTAG72_Tu2
Length of names: 10


In [19]:
onlyIn_sdata = sdata_trmts - adata_trmts
onlyIn_sdata

set()

In [20]:
onlyIn_adata = adata_trmts - sdata_trmts
onlyIn_adata

set()

In [21]:
onlyIn_sdata == onlyIn_sdata

True

#### Spot check for name

In [136]:
treats = [
    "CyPSCA_Tu1",
    "CyPSCA_Tu2",
    "CyTAG72_Tu1",
    "CyTAG72_Tu2",
    "NoTx_Tu1",
    "NoTx_Tu2",
    "RTCyPSCA_Tu1",
    "RTCyPSCA_Tu2",
    "RTCyTAG72_Tu1",
    "RTCyTAG72_Tu2",
]

In [137]:
for i in range(0,10, 1):
    print("")
    print("-----------------------------------------------")
    print(f"Checking treatment: {treats[i]}")
    treats_mask = sdata.obs['treatment'] == treats[i]
    subset_check = sdata[treats_mask].copy()
    
    print(subset_check.obs.loc[:,['treatment', 'condition', 'tumor_loc']].head())


-----------------------------------------------
Checking treatment: CyPSCA_Tu1
                            treatment condition  tumor_loc
F07839_cellid_000000001-1  CyPSCA_Tu1    CyPSCA          1
F07839_cellid_000000002-1  CyPSCA_Tu1    CyPSCA          1
F07839_cellid_000000004-1  CyPSCA_Tu1    CyPSCA          1
F07839_cellid_000000005-1  CyPSCA_Tu1    CyPSCA          1
F07839_cellid_000000006-1  CyPSCA_Tu1    CyPSCA          1

-----------------------------------------------
Checking treatment: CyPSCA_Tu2
                            treatment condition  tumor_loc
F08542_cellid_000047107-1  CyPSCA_Tu2    CyPSCA          2
F08542_cellid_000047110-1  CyPSCA_Tu2    CyPSCA          2
F08542_cellid_000047111-1  CyPSCA_Tu2    CyPSCA          2
F08542_cellid_000047112-1  CyPSCA_Tu2    CyPSCA          2
F08542_cellid_000047113-1  CyPSCA_Tu2    CyPSCA          2

-----------------------------------------------
Checking treatment: CyTAG72_Tu1
                             treatment condition  t

## ENSEMBL

### Two pass

In [215]:
import numpy as np
import pandas as pd
import mygene

mg = mygene.MyGeneInfo()

def extract_ensembl_gene_ids(x):
    """Return a set of ENSMUSG... gene IDs from MyGene 'ensembl' field."""
    if x is None:
        return set()
    if isinstance(x, float) and np.isnan(x):
        return set()
    if isinstance(x, dict):
        g = x.get("gene")
        return {g} if g else set()
    if isinstance(x, list):
        out = set()
        for d in x:
            if isinstance(d, dict) and d.get("gene"):
                out.add(d["gene"])
        return out
    return set()

def resolve_unique_gene_id(df):
    """Given rows for a single query, return a unique gene_id or NaN."""
    genes = set()
    for x in df["ensembl"].tolist():
        genes |= extract_ensembl_gene_ids(x)
    return list(genes)[0] if len(genes) == 1 else np.nan


In [ ]:
symbols = sdata.var_names.tolist()  # your gene symbols

res_sym = mg.querymany(
    symbols,
    scopes="symbol",
    fields="ensembl.gene,symbol",
    species="mouse",
    verbose=True,
)

df_sym = pd.DataFrame(res_sym)

In [216]:
# resolve per query symbol
sym_resolved = (
    df_sym.groupby("query", sort=False)
          .apply(resolve_unique_gene_id)
          .rename("gene_id_sym")
          .reset_index()
)

/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_1987/1851511986.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(resolve_unique_gene_id)


In [217]:
notfound = df_sym.loc[df_sym.get("notfound", False) == True, "query"].dropna().unique().tolist()

res_alias = mg.querymany(
    notfound,
    scopes="alias",
    fields="ensembl.gene,symbol",
    species="mouse",
    verbose=True,
)

df_alias = pd.DataFrame(res_alias)

alias_resolved = (
    df_alias.groupby("query", sort=False)
            .apply(resolve_unique_gene_id)
            .rename("gene_id_alias")
            .reset_index()
)


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
20 input query terms found dup hits:	[('Pih1d3', 2), ('Adss', 2), ('Marc1', 2), ('Marc2', 2), ('Il1f9', 2), ('Lhfp', 2), ('Lor', 2), ('Ol
37 input query terms found no hit:	['Arhgef4-1', 'BC055324', 'BC052040', 'BC029722', 'Pakap-1', 'C87499', 'BC080695', 'C87977', 'C87414
/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_1987/1643389735.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(resolve_unique_gene_id)


In [218]:
resolved = sym_resolved.merge(alias_resolved, on="query", how="left")

# take symbol mapping first; if missing, take alias mapping
resolved["gene_id"] = resolved["gene_id_sym"].combine_first(resolved["gene_id_alias"])

# collision guard: if alias created a gene_id already assigned to another symbol-hit query, drop the alias assignment
symbol_gene_ids = set(resolved.loc[resolved["gene_id_sym"].notna(), "gene_id_sym"])

alias_only_mask = resolved["gene_id_sym"].isna() & resolved["gene_id"].notna()
collides = alias_only_mask & resolved["gene_id"].isin(symbol_gene_ids)
resolved.loc[collides, "gene_id"] = np.nan

resolved = resolved[["query", "gene_id"]]


In [219]:
resolved[resolved["query"].isin(["Cd2", "Ccnd2"])].sort_values("query")


,query,gene_id
6696,Ccnd2,ENSMUSG00000000184
3297,Cd2,ENSMUSG00000027863


In [ ]:
#Checking for bad genes
bad = resolved.loc[
    resolved["gene_id"].notna() & ~resolved["gene_id"].astype(str).str.startswith("ENSMUSG"),
    ["query", "gene_id"]
]
bad.shape, bad.head(20)

In [ ]:
# Removing bad genes
is_ensmusg = resolved["gene_id"].astype("string").str.startswith("ENSMUSG", na=False)
resolved.loc[~is_ensmusg, "gene_id"] = np.nan

In [231]:
# Checking again for bad genes after removal (SC)
bad = resolved.loc[
    resolved["gene_id"].notna() & ~resolved["gene_id"].astype(str).str.startswith("ENSMUSG"),
    ["query", "gene_id"]
]
bad.shape, bad.head(20)

((0, 2),
 Empty DataFrame
 Columns: [query, gene_id]
 Index: [])

In [233]:
# Creating a clean dataset:

symbol_to_gene = dict(zip(resolved["query"], resolved["gene_id"]))
mapped_ids = pd.Index(sdata.var_names).map(symbol_to_gene)

keep = ~pd.isna(mapped_ids)
sdata_c2l = sdata[:, keep].copy()

sdata_c2l.var["SYMBOL"] = sdata_c2l.var_names
sdata_c2l.var_names = pd.Index(mapped_ids[keep])

assert sdata_c2l.var_names.is_unique

#### SC - gene and ensemble ID spot checks

In [243]:
import numpy as np

n = 5

spot = sdata_c2l.var.sample(n=n).copy()
spot["ensembl_gene_id"] = spot.index  # var_names are now Ensembl IDs

spot[["SYMBOL", "ensembl_gene_id"]]


,SYMBOL,ensembl_gene_id
ENSMUSG00000041358,Nutm1,ENSMUSG00000041358
ENSMUSG00000060688,Olfr292,ENSMUSG00000060688
ENSMUSG00000041216,Clvs1,ENSMUSG00000041216
ENSMUSG00000004642,Slbp,ENSMUSG00000004642
ENSMUSG00000030795,Fus,ENSMUSG00000030795


In [244]:
# 1) counts are non-negative integers (or at least non-negative)
assert (sdata_c2l.X.min() >= 0)

# 2) no duplicate gene IDs
assert sdata_c2l.var_names.is_unique


#### SC - Testing mapping results

In [220]:
collisions = resolved.loc[
    resolved["gene_id"].notna() & resolved["gene_id"].duplicated(keep=False)
].sort_values("gene_id")

collisions.shape, collisions.head(30)


((0, 2),
 Empty DataFrame
 Columns: [query, gene_id]
 Index: [])

In [221]:
(collisions.groupby("gene_id")["query"]
           .apply(list)
           .head(20))


Series([], Name: query, dtype: object)

In [230]:
bad = resolved.loc[
    resolved["gene_id"].notna() & ~resolved["gene_id"].astype(str).str.startswith("ENSMUSG"),
    ["query", "gene_id"]
]
bad.shape, bad.head(20)


((0, 2),
 Empty DataFrame
 Columns: [query, gene_id]
 Index: [])

In [229]:
is_ensmusg = resolved["gene_id"].astype("string").str.startswith("ENSMUSG", na=False)
resolved.loc[~is_ensmusg, "gene_id"] = np.nan

In [227]:
map_df[map_df["query"] == "Scaper"][["query","_id","symbol","ensembl","notfound","_score"]].head(20)


,query,_id,symbol,ensembl,notfound,_score
10497,Scaper,244891,Scaper,{'gene': '244891'},NaN,14.738062


In [224]:
symbol_to_gene = dict(zip(resolved["query"], resolved["gene_id"]))
mapped_ids = pd.Index(sdata.var_names).map(symbol_to_gene)

pd.isna(mapped_ids).sum(), pd.Index(mapped_ids.dropna()).duplicated().sum()


(np.int64(111), np.int64(0))

In [225]:
missing_in_resolved = pd.Index(sdata.var_names).difference(pd.Index(resolved["query"]))
len(missing_in_resolved), list(missing_in_resolved[:20])


(0, [])

In [226]:
markers = ["Ptprc","Cd3d","Cd3e","Nkg7","Ms4a1","Lyz2","Adgre1","Col1a1","Epcam","Krt8","Krt18"]
resolved[resolved["query"].isin(markers)].sort_values("query")

,query,gene_id
17060,Adgre1,ENSMUSG00000004730
9953,Cd3d,ENSMUSG00000032094
9954,Cd3e,ENSMUSG00000032093
12477,Col1a1,ENSMUSG00000001506
17190,Epcam,ENSMUSG00000045394
15628,Krt18,ENSMUSG00000023043
15627,Krt8,ENSMUSG00000049382
11350,Lyz2,ENSMUSG00000069516
17910,Ms4a1,ENSMUSG00000024673
7494,Nkg7,ENSMUSG00000004612


## One Pass

In [99]:
def ensemble_to_gene_id(x):
    # if x is None:
    if pd.isna(x):
        return np.nan
    if isinstance(x, dict):
        return x.get("gene", np.nan)
        
    return type(x)

In [11]:
sdata.var['SYMBOL'] = sdata.var_names

In [108]:
symbols = pd.Index(sdata.var_names).astype(str).unique().tolist()

print("n genes:", len(symbols))
print("example:", symbols[:10])

n genes: 19059
example: ['Xkr4', 'Rp1', 'Sox17', 'Lypla1', 'Tcea1', 'Rgs20', 'Atp6v1h', 'Oprk1', 'Npbwr1', 'Rb1cc1']


In [119]:
mg = mygene.MyGeneInfo()

res = mg.querymany(
    symbols,
    scopes=["symbol", "alias"],
    fields="ensembl.gene,symbol",
    species="mouse",
    verbose = True
)

Input sequence provided is already in string format. No operation performed
758 input query terms found dup hits:	[('Rp1', 4), ('Tram1', 2), ('Defb41', 2), ('Khdc1c', 2), ('Khdc1b', 2), ('Pih1d3', 2), ('Gls', 2), (
37 input query terms found no hit:	['Arhgef4-1', 'BC055324', 'BC052040', 'BC029722', 'Pakap-1', 'C87499', 'BC080695', 'C87977', 'C87414


In [123]:
# Converting list of dicts into 
map_df = pd.DataFrame(res)

# Saving Missing Genes
missing_map_df = map_df[map_df['notfound'] == True]
missingCSV = results_folder / "missing_sdata_genes.csv"
missing_map_df.to_csv(missingCSV)

In [161]:
# grabbing the the ensembl ID's
map_df['gene_id'] = map_df['ensembl'].apply(ensemble_to_gene_id)

map_df.head()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

### Testing

#### Recommended method for dealing with missing ensembls

In [ ]:
np.where()

In [ ]:
import numpy as np
import pandas as pd

# map_df must have columns: 'query' (symbol) and 'gene_id' (Ensembl or NaN)
# where 'query' corresponds to your current sdata.var_names (symbols)
sym2ens = dict(zip(map_df["query"].astype(str), map_df["gene_id"]))

var = sdata.var.copy()

# Keep original symbols explicitly
var["SYMBOL"] = sdata.var_names.astype(str)

# Add Ensembl mapping column
var["gene_id"] = var["SYMBOL"].map(sym2ens)

# Build a new, collision-safe var_names:
# - mapped -> Ensembl
# - unmapped -> prefixed symbol
new_names = np.where(
    var["gene_id"].notna(),
    var["gene_id"].astype(str).values,
    ("SYMBOL:" + var["SYMBOL"].astype(str)).values
)

# Ensure uniqueness (absolutely do this)
new_names = pd.Index(new_names)
if not new_names.is_unique:
    # minimal deterministic fix
    new_names = pd.Index(pd.io.parsers.ParserBase({'names': new_names})._maybe_dedup_names(new_names))

# Write back
sdata.var = var
sdata.var_names = new_names

# Sanity checks
print("n genes:", sdata.n_vars)
print("mapped:", sdata.var["gene_id"].notna().sum())
print("unmapped:", sdata.var["gene_id"].isna().sum())
print("unique var_names:", sdata.var_names.is_unique)
print("example var_names:", sdata.var_names[:10].tolist())


What this means for cell2location

If your reference is Ensembl-indexed (common): your unmapped genes (now SYMBOL:...) simply won’t overlap and won’t affect fitting.

If your reference is symbol-indexed: then you should do the analogous thing on the reference (map to Ensembl and namespace unmapped) so both end up in the same feature namespace. Otherwise you’ll get low overlap or inconsistent matching.

My recommendation for your workflow

Do the “safe mixed-ID” approach above only if you genuinely want to keep all genes in-place. It’s reasonable.

But be clear-eyed: those unmapped genes are almost never going to help deconvolution unless they also appear cleanly in the reference with the same identifiers. Their value is more for downstream biology once you have annotations.

If you paste:

whether your reference adata_ref.var_names are Ensembl or symbols

and len(set(adata_ref.var_names) & set(sdata.var_names)) after this step
I can tell you immediately if your gene overlap is healthy or if you’re about to starve the model of shared genes.

#### Testing Function

In [58]:
map_df.head()

,query,_id,_score,ensembl,symbol,notfound
0,Xkr4,497097,14.785570,{'gene': 'ENSMUSG00000051951'},Xkr4,NaN
1,Rp1,19888,15.440001,{'gene': 'ENSMUSG00000025900'},Rp1,NaN
2,Sox17,20671,15.149700,{'gene': 'ENSMUSG00000025902'},Sox17,NaN
3,Lypla1,18777,14.852126,{'gene': 'ENSMUSG00000025903'},Lypla1,NaN
4,Tcea1,21399,14.743513,{'gene': 'ENSMUSG00000033813'},Tcea1,NaN


In [84]:
na_df     = map_df.loc[map_df['ensembl'].isna()].sample(n=5, random_state=0)
not_na_df = map_df.loc[map_df['ensembl'].notna()].sample(n=5, random_state=0)

combined = pd.concat([na_df, not_na_df], axis=0)

combined

,query,_id,_score,ensembl,symbol,notfound
1840,Olfr1217,NaN,NaN,NaN,NaN,True
15545,Olfr288,NaN,NaN,NaN,NaN,True
1190,Il1f5,NaN,NaN,NaN,NaN,True
1739,Olfr1097,NaN,NaN,NaN,NaN,True
14562,Olfr732,NaN,NaN,NaN,NaN,True
15256,Oplah,75475,14.843104,{'gene': 'ENSMUSG00000022562'},Oplah,NaN
12599,Tns4,217169,15.284526,{'gene': 'ENSMUSG00000017607'},Tns4,NaN
13188,Arhgap5,11855,14.764845,{'gene': 'ENSMUSG00000035133'},Arhgap5,NaN
9419,Znrf1,170737,14.768924,{'gene': 'ENSMUSG00000033545'},Znrf1,NaN
758,Tor1aip2,240832,16.388847,{'gene': 'ENSMUSG00000050565'},Tor1aip2,NaN


In [ ]:
x.get()

In [145]:
def testing_NA_check(x):
    # if x is None:
    if pd.isna(x):
        return "Oi, me blank"
        # return np.nan
    if isinstance(x, dict):
        return x.get("gene", np.nan)
        
    return type(x)

In [146]:
combined['TEST'] = combined['ensembl'].apply(testing_NA_check)
combined

,query,_id,_score,ensembl,symbol,notfound,TEST
1840,Olfr1217,NaN,NaN,NaN,NaN,True,"Oi, me blank"
15545,Olfr288,NaN,NaN,NaN,NaN,True,"Oi, me blank"
1190,Il1f5,NaN,NaN,NaN,NaN,True,"Oi, me blank"
1739,Olfr1097,NaN,NaN,NaN,NaN,True,"Oi, me blank"
14562,Olfr732,NaN,NaN,NaN,NaN,True,"Oi, me blank"
15256,Oplah,75475,14.843104,{'gene': 'ENSMUSG00000022562'},Oplah,NaN,ENSMUSG00000022562
12599,Tns4,217169,15.284526,{'gene': 'ENSMUSG00000017607'},Tns4,NaN,ENSMUSG00000017607
13188,Arhgap5,11855,14.764845,{'gene': 'ENSMUSG00000035133'},Arhgap5,NaN,ENSMUSG00000035133
9419,Znrf1,170737,14.768924,{'gene': 'ENSMUSG00000033545'},Znrf1,NaN,ENSMUSG00000033545
758,Tor1aip2,240832,16.388847,{'gene': 'ENSMUSG00000050565'},Tor1aip2,NaN,ENSMUSG00000050565


In [75]:
NoMatch = map_df.loc[map_df['ensembl'].isna()].copy()
NoMatch.head()

,query,_id,_score,ensembl,symbol,notfound
80,Pih1d3,NaN,NaN,NaN,NaN,True
96,Arhgef4-1,NaN,NaN,NaN,NaN,True
223,Fam126b,NaN,NaN,NaN,NaN,True
255,Gpr1,NaN,NaN,NaN,NaN,True
296,March4,NaN,NaN,NaN,NaN,True


In [76]:
NoMatch['TEST'] = NoMatch['ensembl'].apply(testing_NA_check).copy()
NoMatch.head()

,query,_id,_score,ensembl,symbol,notfound,TEST
80,Pih1d3,NaN,NaN,NaN,NaN,True,NaN
96,Arhgef4-1,NaN,NaN,NaN,NaN,True,NaN
223,Fam126b,NaN,NaN,NaN,NaN,True,NaN
255,Gpr1,NaN,NaN,NaN,NaN,True,NaN
296,March4,NaN,NaN,NaN,NaN,True,NaN


In [ ]:
map_df[map_df['ensembl'] == np.nan]

In [62]:
isinstance(map_df['ensembl'][3], dict)

True

#### Understanding ensemble column

In [73]:

map_df.loc[3:5,:]

,query,_id,_score,ensembl,symbol,notfound
3,Lypla1,18777,14.852126,{'gene': 'ENSMUSG00000025903'},Lypla1,NaN
4,Tcea1,21399,14.743513,{'gene': 'ENSMUSG00000033813'},Tcea1,NaN
5,Rgs20,58175,14.810723,{'gene': 'ENSMUSG00000002459'},Rgs20,NaN


In [60]:
map_df['ensembl']

0        {'gene': 'ENSMUSG00000051951'}
1        {'gene': 'ENSMUSG00000025900'}
2        {'gene': 'ENSMUSG00000025902'}
3        {'gene': 'ENSMUSG00000025903'}
4        {'gene': 'ENSMUSG00000033813'}
                      ...              
19087    {'gene': 'ENSMUSG00000064363'}
19088    {'gene': 'ENSMUSG00000064367'}
19089    {'gene': 'ENSMUSG00000064368'}
19090    {'gene': 'ENSMUSG00000064370'}
19091                               NaN
Name: ensembl, Length: 19092, dtype: object

In [15]:
type(map_df['ensembl'][0])

dict

In [16]:
map_df['ensembl'][0]['gene']

'ENSMUSG00000051951'

In [17]:
map_df['ensembl'][0].get('gene')

'ENSMUSG00000051951'

In [17]:
map_df['ensembl'][0]

'ENSMUSG00000051951'

#### Missing Ensembl ID's

In [121]:
map_df[map_df['notfound'] == True]

,query,_id,_score,ensembl,symbol,notfound
80,Pih1d3,NaN,NaN,NaN,NaN,True
96,Arhgef4-1,NaN,NaN,NaN,NaN,True
223,Fam126b,NaN,NaN,NaN,NaN,True
255,Gpr1,NaN,NaN,NaN,NaN,True
296,March4,NaN,NaN,NaN,NaN,True
...,...,...,...,...,...,...
18882,Prame,NaN,NaN,NaN,NaN,True
18908,H2bfm,NaN,NaN,NaN,NaN,True
18927,Pih1h3b,NaN,NaN,NaN,NaN,True
18940,Kcne1l,NaN,NaN,NaN,NaN,True


In [111]:
map_df[map_df['ensembl'].isna()]['query'].tolist()

['Pih1d3',
 'Arhgef4-1',
 'Fam126b',
 'Gpr1',
 'March4',
 'BC035947',
 'Krtap28-10',
 'Iqca',
 'Olfr1416',
 'Olfr1415',
 'Olfr1414',
 'Olfr1413',
 'Olfr1412',
 'Olfr1411',
 'Olfr1410',
 'Olfr12',
 'Sept2',
 'D1Ertd622e',
 'Dars',
 'Chil1',
 'Fam129a',
 'Eef1aknmt',
 'Mettl11b',
 'BC055324',
 'Dusp27',
 'Olfr16',
 'Olfr1408',
 'Olfr1406',
 'Olfr218',
 'Olfr1404',
 'Olfr418',
 'Olfr433',
 'Olfr432',
 'Olfr430',
 'Olfr429',
 'Olfr427',
 'Olfr231',
 'Olfr424',
 'Olfr420',
 'Olfr419',
 'Olfr417',
 'Olfr248',
 'Olfr414',
 'Olfr220',
 'Rbm8a2',
 'Adss',
 'Marc1',
 'Marc2',
 'Eprs',
 'Fam71a',
 'Atp5c1',
 'Il1f8',
 'Il1f9',
 'Il1f6',
 'Il1f5',
 'Fam166a',
 'Tmem250-ps',
 'Wdr34',
 'Fam102a',
 'Fam129b',
 'Olfr338',
 'Olfr339',
 'Olfr340',
 'Olfr341',
 'Olfr342',
 'Olfr344',
 'Olfr345',
 'Olfr346',
 'Olfr347',
 'Olfr348',
 'Olfr50',
 'Olfr3',
 'Olfr350',
 'Olfr351',
 'Olfr352',
 'Olfr353',
 'Olfr354',
 'Olfr355',
 'Olfr356',
 'Olfr357',
 'Olfr358',
 'Olfr360',
 'Olfr361',
 'Olfr362',
 'Olfr364-

In [128]:
adata_ref.obs

,orig.ident,nCount_RNA,nFeature_RNA,Barcode,Library,condition,file_name,percent.mt,unintegrated_clusters,seurat_clusters,harmony_clusters,sctype_classification,seurat.cluster.ann,treatment,cell_type,UMAP_R_1,UMAP_R_2
56948_AAACGAACACTAGGTT-1,NoTx,2142,851,AAACGAACACTAGGTT-1,56948,NoTx,NoTx_Tu1_1,9.290383,10,9,9,Neutrophils,9: Neutrophils,NoTx_Tu1,Neutrophils,5.049781,-9.935757
56948_AAACGCTGTTGGGTTT-1,NoTx,1897,658,AAACGCTGTTGGGTTT-1,56948,NoTx,NoTx_Tu1_1,5.640485,10,9,9,Neutrophils,9: Neutrophils,NoTx_Tu1,Neutrophils,6.671258,-9.771027
56948_AAACGCTTCCTCGATC-1,NoTx,16414,3769,AAACGCTTCCTCGATC-1,56948,NoTx,NoTx_Tu1_1,4.563178,6,3,3,Macrophage,3: Macrophage,NoTx_Tu1,Macrophage,7.841278,3.778570
56948_AAAGAACCAATACAGA-1,NoTx,8973,2987,AAAGAACCAATACAGA-1,56948,NoTx,NoTx_Tu1_1,2.084030,6,3,3,Macrophage,3: Macrophage,NoTx_Tu1,Macrophage,8.648595,2.988089
56948_AAAGAACCACAGGATG-1,NoTx,693,452,AAAGAACCACAGGATG-1,56948,NoTx,NoTx_Tu1_1,16.161616,3,2,2,Fibroblast,2: Fibroblast,NoTx_Tu1,Fibroblast,-4.163362,-6.711728
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56977_TTTGTTGCACACACTA-1,RTCyPSCA,7529,2635,TTTGTTGCACACACTA-1,56977,RTCyPSCA,RTCyPSCA_Tu2_3,7.371497,0,1,1,Monocyte,1: Monocyte,RTCyPSCA_Tu2,Monocyte,-1.058346,3.443820
56977_TTTGTTGCACTCTAGA-1,RTCyPSCA,4276,1797,TTTGTTGCACTCTAGA-1,56977,RTCyPSCA,RTCyPSCA_Tu2_3,8.185220,0,0,0,Monocyte,0: Monocyte,RTCyPSCA_Tu2,Monocyte,1.897415,3.340785
56977_TTTGTTGCATCTTTCA-1,RTCyPSCA,1083,493,TTTGTTGCATCTTTCA-1,56977,RTCyPSCA,RTCyPSCA_Tu2_3,11.357341,5,4,4,Macrophage,4: Macrophage,RTCyPSCA_Tu2,Macrophage,-1.845459,-3.558179
56977_TTTGTTGTCAGCTGTA-1,RTCyPSCA,7816,2784,TTTGTTGTCAGCTGTA-1,56977,RTCyPSCA,RTCyPSCA_Tu2_3,2.712385,0,16,16,Fibroblast,16: Fibroblast,RTCyPSCA_Tu2,Fibroblast,-0.701925,-9.077349


map_df]

In [124]:
map_df[(map_df['ensembl'].isna()) & (map_df['query'] == 'Chil1')]

,query,_id,_score,ensembl,symbol,notfound


In [131]:
map_df[(map_df['ensembl'].isna()) & (map_df['query'] == 'Arhgef4')]

,query,_id,_score,ensembl,symbol,notfound


In [133]:
items = ["apple_pie", "banana_split", "cherry_tart", "pineapple"]
query = "apple"

matches = [s for s in items if query in s]

In [140]:
# symbos
# Arhgef4-1
# Pakap-1
# Fam220a-1
# Aldoa-1
# Pcdha11-1
# Fam90a1b-1

query = ["Arhgef4", "Pakap", "Fam220a", "Aldoa", "Pcdha11", "Fam90a1b"]

matches = [s for s in symbols if any(q in s for q in query)]

In [141]:
matches

['Arhgef4',
 'Arhgef4-1',
 'Pakap',
 'Pakap-1',
 'Aldoart1',
 'Fam220a',
 'Fam220a-1',
 'Aldoa',
 'Aldoa-1',
 'Aldoart2',
 'Arhgef40',
 'Pcdha11',
 'Pcdha11-1',
 'Fam90a1b',
 'Fam90a1b-1']

#### Different types ensembl column

In [151]:
map_df["ensembl"].apply(lambda x: type(x).__name__).value_counts()

ensembl
dict     19745
float      148
list        15
Name: count, dtype: int64

In [162]:
map_df.loc[
    map_df['ensembl'].apply(lambda x: isinstance(x, list)),
    ['query', 'ensembl']
]

,query,ensembl
1264,Ndor1,"[{'gene': 'ENSMUSG00000006471'}, {'gene': 'ENS..."
1421,Fam78a,"[{'gene': 'ENSMUSG00000050592'}, {'gene': 'ENS..."
3121,Arfip1,"[{'gene': 'ENSMUSG00000102805'}, {'gene': 'ENS..."
3718,Samd13,"[{'gene': 'ENSMUSG00000090202'}, {'gene': 'ENS..."
4048,Pakap,"[{'gene': 'ENSMUSG00000038729'}, {'gene': 'ENS..."
6021,Pms2,"[{'gene': 'ENSMUSG00000075569'}, {'gene': 'ENS..."
8317,Pde2a,"[{'gene': 'ENSMUSG00000030653'}, {'gene': 'ENS..."
8498,Timm10b,"[{'gene': 'ENSMUSG00000089847'}, {'gene': 'ENS..."
9221,Poteg,"[{'gene': 'ENSMUSG00000063932'}, {'gene': 'ENS..."
9763,Dpep2,"[{'gene': 'ENSMUSG00000053687'}, {'gene': 'ENS..."


In [164]:
map_df.loc[map_df['query'] == 'Ndor1']['ensembl']

1263                       {'gene': 'ENSMUSG00000115018'}
1264    [{'gene': 'ENSMUSG00000006471'}, {'gene': 'ENS...
Name: ensembl, dtype: object

In [156]:
map_df.loc[
    map_df['ensembl'].apply(lambda x: isinstance(x, float) and np.isnan(x)),
    ['query', 'ensembl']
]

,query,ensembl
104,Arhgef4-1,NaN
213,Mob4,NaN
376,BC035947,NaN
398,Krtap28-10,NaN
403,Pid1,NaN
...,...,...
19521,Fam90a1b-1,NaN
19532,Pfn5,NaN
19818,Sms,NaN
19832,Rs1,NaN


In [165]:
def classify_ensembl_entry(x):
    if x is None:
        return "None"
    if isinstance(x, float) and np.isnan(x):
        return "NaN"
    if isinstance(x, dict):
        return "dict_single"
    if isinstance(x, list):
        return "list_multiple"
    return "other"

map_df["ensembl_type"] = map_df["ensembl"].apply(classify_ensembl_entry)
map_df["ensembl_type"].value_counts()


ensembl_type
dict_single      19745
NaN                148
list_multiple       15
Name: count, dtype: int64

In [167]:
map_df

,query,_id,_score,ensembl,symbol,notfound,ensembl_type
0,Xkr4,497097,14.789040,{'gene': 'ENSMUSG00000051951'},Xkr4,NaN,dict_single
1,Rp1,54402,15.918115,{'gene': 'ENSMUSG00000061207'},Stk19,NaN,dict_single
2,Rp1,212307,15.918115,{'gene': 'ENSMUSG00000024277'},Mapre2,NaN,dict_single
3,Rp1,13117,15.918115,{'gene': 'ENSMUSG00000066072'},Cyp4a10,NaN,dict_single
4,Rp1,19888,15.440001,{'gene': 'ENSMUSG00000025900'},Rp1,NaN,dict_single
...,...,...,...,...,...,...,...
19903,mt-Nd4,17719,19.825235,{'gene': 'ENSMUSG00000064363'},mt-Nd4,NaN,dict_single
19904,mt-Nd5,17721,19.069921,{'gene': 'ENSMUSG00000064367'},mt-Nd5,NaN,dict_single
19905,mt-Nd6,17722,19.605286,{'gene': 'ENSMUSG00000064368'},mt-Nd6,NaN,dict_single
19906,mt-Cytb,17711,21.511091,{'gene': 'ENSMUSG00000064370'},mt-Cytb,NaN,dict_single


In [193]:
696/18301

0.03803070870444238

In [192]:
query_hit_counts = map_df.groupby("query").size()

query_hit_counts.value_counts().sort_index()


1    18301
2      696
3       45
4       11
5        3
6        2
9        1
Name: count, dtype: int64

In [190]:
n_total = resolved.shape[0]
n_unique = resolved["gene_id"].notna().sum()
n_ambiguous = resolved["gene_id"].isna().sum()

print("Total queries:", n_total)
print("Uniquely resolved:", n_unique)
print("Ambiguous / unresolved:", n_ambiguous)


NameError: name 'resolved' is not defined

In [169]:
for t in map_df["ensembl_type"].unique():
    print(f"\n=== {t} ===")
    display(
        map_df.loc[map_df["ensembl_type"] == t, ["query", "ensembl", "symbol"]].head(5)
    )


=== dict_single ===


,query,ensembl,symbol
0,Xkr4,{'gene': 'ENSMUSG00000051951'},Xkr4
1,Rp1,{'gene': 'ENSMUSG00000061207'},Stk19
2,Rp1,{'gene': 'ENSMUSG00000024277'},Mapre2
3,Rp1,{'gene': 'ENSMUSG00000066072'},Cyp4a10
4,Rp1,{'gene': 'ENSMUSG00000025900'},Rp1



=== NaN ===


,query,ensembl,symbol
104,Arhgef4-1,NaN,NaN
213,Mob4,NaN,Mobq4
376,BC035947,NaN,BC035947
398,Krtap28-10,NaN,Krtap28-10
403,Pid1,NaN,Prid1



=== list_multiple ===


,query,ensembl,symbol
1264,Ndor1,"[{'gene': 'ENSMUSG00000006471'}, {'gene': 'ENS...",Ndor1
1421,Fam78a,"[{'gene': 'ENSMUSG00000050592'}, {'gene': 'ENS...",Fam78a
3121,Arfip1,"[{'gene': 'ENSMUSG00000102805'}, {'gene': 'ENS...",Arfip1
3718,Samd13,"[{'gene': 'ENSMUSG00000090202'}, {'gene': 'ENS...",Samd13
4048,Pakap,"[{'gene': 'ENSMUSG00000038729'}, {'gene': 'ENS...",Pakap


In [185]:
a = map_df[map_df['query'] == 'Ndor1']['ensembl']

In [189]:
map_df[map_df['query'] == 'Ndor1']

,query,_id,_score,ensembl,symbol,notfound,ensembl_type
1263,Ndor1,ENSMUSG00000115018,14.761046,{'gene': 'ENSMUSG00000115018'},Ndor1,NaN,dict_single
1264,Ndor1,78797,14.761046,"[{'gene': 'ENSMUSG00000006471'}, {'gene': 'ENS...",Ndor1,NaN,list_multiple


In [188]:
a

1263                       {'gene': 'ENSMUSG00000115018'}
1264    [{'gene': 'ENSMUSG00000006471'}, {'gene': 'ENS...
Name: ensembl, dtype: object

In [187]:
a.loc[1264]

[{'gene': 'ENSMUSG00000006471'}, {'gene': 'ENSMUSG00000115074'}]

#### Resulving everything

In [194]:
import numpy as np
import pandas as pd

def extract_genes(ensembl_field):
    # returns a list of ENSMUSG... ids (possibly empty)
    if ensembl_field is None:
        return []
    if isinstance(ensembl_field, float) and np.isnan(ensembl_field):
        return []
    if isinstance(ensembl_field, dict):
        g = ensembl_field.get("gene")
        return [g] if g else []
    if isinstance(ensembl_field, list):
        genes = []
        for d in ensembl_field:
            if isinstance(d, dict) and d.get("gene"):
                genes.append(d["gene"])
        return genes
    return []


In [195]:
def resolve_query_group(g):
    """
    g: subset of map_df for one query symbol
    returns: resolved Ensembl gene_id or np.nan
    """
    # Prefer dict_single hits (unambiguous)
    dict_hits = g[g["ensembl"].apply(lambda x: isinstance(x, dict))].copy()
    if len(dict_hits) > 0:
        # pick best score among dict hits
        best = dict_hits.sort_values("_score", ascending=False).iloc[0]
        genes = extract_genes(best["ensembl"])
        return genes[0] if len(genes) == 1 else np.nan

    # Otherwise consider list hits
    list_hits = g[g["ensembl"].apply(lambda x: isinstance(x, list))].copy()
    if len(list_hits) > 0:
        # collect all genes across all list hits
        all_genes = []
        for x in list_hits["ensembl"]:
            all_genes.extend(extract_genes(x))
        all_genes = sorted(set(all_genes))
        return all_genes[0] if len(all_genes) == 1 else np.nan

    # No hits
    return np.nan

resolved = (
    map_df.groupby("query", sort=False)
          .apply(resolve_query_group)
          .rename("gene_id")
          .reset_index()
)

resolved.head()


/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_1987/56271029.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(resolve_query_group)


,query,gene_id
0,Xkr4,ENSMUSG00000051951
1,Rp1,ENSMUSG00000061207
2,Sox17,ENSMUSG00000025902
3,Lypla1,ENSMUSG00000025903
4,Tcea1,ENSMUSG00000033813


In [196]:
resolved[resolved["query"] == "Ndor1"]

,query,gene_id
1211,Ndor1,ENSMUSG00000115018


In [200]:
resolved["gene_id"].duplicated().sum()

np.int64(556)

In [199]:
resolved.loc[resolved["gene_id"].duplicated(keep=False)].sort_values("gene_id")

,query,gene_id
3297,Cd2,ENSMUSG00000000184
6696,Ccnd2,ENSMUSG00000000184
7493,Lim2,ENSMUSG00000000247
1464,Lhx2,ENSMUSG00000000247
11962,Tom1l2,ENSMUSG00000000538
...,...,...
18502,Tex13c2,NaN
18650,Olfr1326-ps1,NaN
18697,Fam90a1b,NaN
18698,Fam90a1b-1,NaN


In [201]:
 resolved.groupby("query")["gene_id"].nunique().max()

np.int64(1)

In [202]:
resolved["gene_id"].notna().mean()

np.float64(0.995959913951414)

In [203]:
resolved.loc[resolved["gene_id"].isna(), "query"].sort_values().tolist()

['AC150683.1',
 'AC151602.1',
 'AF067061',
 'AF067063',
 'AF366264',
 'AI413582',
 'AU022751',
 'AW549877',
 'AW822073',
 'AY761184',
 'Aldoa-1',
 'Arfip1',
 'Arhgef4-1',
 'BC003965',
 'BC005561',
 'BC017158',
 'BC024978',
 'BC029722',
 'BC030867',
 'BC049352',
 'BC049762',
 'BC051142',
 'BC052040',
 'BC053393',
 'BC055324',
 'BC067074',
 'BC080695',
 'BC147527',
 'C87414',
 'C87499',
 'C87977',
 'CN725425',
 'Dpep2',
 'FQ976806.1',
 'Fam220a-1',
 'Fam78a',
 'Fam90a1b',
 'Fam90a1b-1',
 'Ftl1-ps1',
 'Gcat',
 'Lsamp',
 'Nme9',
 'OR5BS1P',
 'Olfr1025-ps1',
 'Olfr104-ps',
 'Olfr105-ps',
 'Olfr106-ps',
 'Olfr1174-ps',
 'Olfr1175-ps',
 'Olfr1191-ps1',
 'Olfr1291-ps1',
 'Olfr1326-ps1',
 'Olfr1438-ps1',
 'Olfr1493-ps1',
 'Olfr1555-ps1',
 'Olfr465-ps1',
 'Olfr680-ps1',
 'Olfr718-ps1',
 'Olfr721-ps1',
 'Olfr764-ps1',
 'Olfr766-ps1',
 'Olfr839-ps1',
 'Olfr896-ps1',
 'Pakap',
 'Pakap-1',
 'Pcdha11-1',
 'Pde2a',
 'Pms2',
 'Poteg',
 'Rsph10b',
 'Samd13',
 'Tcrg-V6',
 'Tex13c2',
 'Timm10b',
 'Vamp7',

In [204]:
# assuming sdata.X is counts
import numpy as np

symbol_to_resolved = dict(zip(resolved["query"], resolved["gene_id"]))

gene_is_unresolved = np.array([
    symbol_to_resolved.get(g, np.nan) is np.nan
    for g in sdata.var_names
])

expr_total = sdata.X.sum()
expr_unresolved = sdata.X[:, gene_is_unresolved].sum()

expr_unresolved / expr_total


np.float32(0.0006681746)

In [207]:
resolved[resolved["query"].str.startswith("mt-", na=False)].head()

,query,gene_id
19045,mt-Nd1,ENSMUSG00000064341
19046,mt-Nd2,ENSMUSG00000064345
19047,mt-Co1,ENSMUSG00000064351
19048,mt-Co2,ENSMUSG00000064354
19049,mt-Atp8,ENSMUSG00000064356


In [208]:
resolved[resolved["query"].str.match(r"^Rpl|^Rps", na=False)].head()

,query,gene_id


In [209]:
sdata.var_names.is_unique

True

In [210]:
assert "SYMBOL" in sdata.var.columns


In [211]:
overlap = len(set(adata_ref.var_names) & set(sdata.var_names))
print("Gene overlap:", overlap)

Gene overlap: 16342


In [212]:
for g in ["Rp1", "Ndor1", "Scnm1"]:
    print(resolved[resolved["query"] == g])


  query             gene_id
1   Rp1  ENSMUSG00000061207
      query             gene_id
1211  Ndor1  ENSMUSG00000115018
      query             gene_id
3203  Scnm1  ENSMUSG00000092607


In [213]:
resolved.loc[
    resolved["gene_id"].notna() & resolved["gene_id"].duplicated(keep=False)
].sort_values("gene_id")


,query,gene_id
3297,Cd2,ENSMUSG00000000184
6696,Ccnd2,ENSMUSG00000000184
7493,Lim2,ENSMUSG00000000247
1464,Lhx2,ENSMUSG00000000247
11962,Tom1l2,ENSMUSG00000000538
...,...,...
1437,Olfr50,ENSMUSG00000111021
17960,Cntf,ENSMUSG00000118491
17961,Zfp91,ENSMUSG00000118491
7698,A26c2,ENSMUSG00000142950


# Cell2location 

I have to consider the batch key when using cell to location, should I use TMA, replicate numbers? 